# РАСЧЕТНО-ГРАФИЧЕСКАЯ РАБОТА ПО ДИСЦИПЛИНЕ «МАШИННОЕ ОБУЧЕНИЕ И БОЛЬШИЕ ДАННЫЕ».
# Тема: «Разработка Web-приложения (дашборда) для инференса (вывода) моделей ML и анализа данных»

Выполнил: Мазунин Даниил, ФИТ-242

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, BaggingRegressor, StackingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
import lightgbm as lgb
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
import joblib

In [2]:
df = pd.read_csv('data/house_regression.csv')

In [3]:
X = df.drop(columns=['price_log'])
y = df['price_log']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

joblib.dump(scaler, "models/scaler.pkl")

['models/scaler.pkl']

# ML1

In [ ]:
poly_ridge = Pipeline([
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('ridge', Ridge(random_state=42))
])

poly_ridge.fit(X_train, y_train)
y_pred_ridge = poly_ridge.predict(X_test)

print('ML1 - Polynomial Ridge Regression')
print('R2:', r2_score(y_test, y_pred_ridge))
print('MAE:', mean_absolute_error(y_test, y_pred_ridge))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred_ridge)))

joblib.dump(poly_ridge, 'models/poly_ridge_model.pkl')

ML1 - Polynomial Ridge Regression
R2: 0.8190839138315141
MAE: 0.16779961418315983
RMSE: 0.22708342223180672


c:\Users\DaniilMS\miniconda3\envs\lab4_1\Lib\site-packages\sklearn\linear_model\_ridge.py:211: LinAlgWarning: Ill-conditioned matrix (rcond=1.57182e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


['poly_ridge_model.pkl']

# ML2

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold
from scipy.stats import randint, uniform

gb_random = {
    'n_estimators': randint(50, 400),
    'learning_rate': uniform(0.005, 0.295),
    'max_depth': randint(3, 15),
    'subsample': uniform(0.5, 0.5),
    'min_samples_split': randint(2, 20)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
gb_model = RandomizedSearchCV(GradientBoostingRegressor(random_state=42), gb_random, n_iter=15, cv=kf, scoring='neg_mean_absolute_error', random_state=42)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

print('ML2 - Gradient Boosting')
print('R2:', r2_score(y_test, y_pred_gb))
print('MAE:', mean_absolute_error(y_test, y_pred_gb))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred_gb)))

joblib.dump(gb_model.best_estimator_, 'models/gb_model.pkl')

ML2 - Gradient Boosting
R2: 0.9096696293227934
MAE: 0.11566236142487271
RMSE: 0.1604588719888921


['gb_model.pkl']

# ML3

In [ ]:
from lightgbm import LGBMRegressor

lgb_random = {
    'n_estimators': randint(50, 500),
    'learning_rate': uniform(0.005, 0.295),
    'max_depth': randint(-1, 10),
    'num_leaves': randint(20, 150),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'reg_alpha': uniform(0, 10),
    'reg_lambda': uniform(0, 10)
}

lgb_model = RandomizedSearchCV(LGBMRegressor(random_state=42, verbose=-1), lgb_random, n_iter=15, cv=kf,
                            scoring='neg_mean_absolute_error', random_state=42)
lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test)

print('ML3 - LightGBM')
print('R2:', r2_score(y_test, y_pred_lgb))
print('MAE:', mean_absolute_error(y_test, y_pred_lgb))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred_lgb)))

lgb_model.best_estimator_.booster_.save_model('models/lgb_model.txt')

c:\Users\DaniilMS\miniconda3\envs\lab4_1\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Не удается найти указанный файл
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\DaniilMS\miniconda3\envs\lab4_1\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\DaniilMS\miniconda3\envs\lab4_1\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DaniilMS\miniconda3\envs\lab4_1\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\DaniilMS\miniconda3\envs\lab4_1\Lib\subprocess.py",

ML3 - LightGBM
R2: 0.9119846636131809
MAE: 0.11310547352802805
RMSE: 0.15838936409789414


# ML4

In [ ]:
from sklearn.model_selection import GridSearchCV

bg_grid = {
    'n_estimators': [10, 20, 50],
    'max_samples': [0.5, 0.8, 1.0],
    'max_features': [0.5, 0.8, 1.0]
}

bag_model = GridSearchCV(BaggingRegressor(random_state=42), bg_grid, cv=kf, scoring='neg_mean_absolute_error')
bag_model.fit(X_train, y_train)
y_pred_bag = bag_model.predict(X_test)

print('ML4 - Bagging')
print('R2:', r2_score(y_test, y_pred_bag))
print('MAE:', mean_absolute_error(y_test, y_pred_bag))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred_bag)))

joblib.dump(bag_model.best_estimator_, 'models/bag_model.pkl')

ML4 - Bagging
R2: 0.8882761136691926
MAE: 0.12764842533953158
RMSE: 0.17845135739308715


['bag_model.pkl']

# ML5

In [ ]:
base_estimators = [
    ('rf', RandomForestRegressor(n_estimators=50, random_state=42)),
    ('gb', GradientBoostingRegressor(n_estimators=50, random_state=42)),
    ('ridge', Ridge(random_state=42))
]

stack_param_grid = {
    'final_estimator__alpha': [0.1, 1.0, 10.0],
    'cv': [3, 5]
}

stack_model = GridSearchCV(
    StackingRegressor(estimators=base_estimators, final_estimator=Ridge(random_state=42)),
    stack_param_grid, cv=kf, scoring='neg_mean_absolute_error'
)
stack_model.fit(X_train, y_train)
y_pred_stack = stack_model.predict(X_test)

print('ML5 - Stacking')
print('R2:', r2_score(y_test, y_pred_stack))
print('MAE:', mean_absolute_error(y_test, y_pred_stack))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred_stack)))

joblib.dump(stack_model.best_estimator_, 'models/stack_model.pkl')

ML5 - Stacking
R2: 0.8898792164972096
MAE: 0.12806142336043208
RMSE: 0.1771664508028666


['stack_model.pkl']

# ML6

In [5]:
def build_kt_model(hp):
    units = hp.Choice('units', [32, 64, 128])
    model = Sequential([
        Input(shape=(X_train_sc.shape[1],)),
        Dense(units, activation='relu'),
        Dense(units // 2, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam' , loss='mse')
    return model
    
tuner = kt.RandomSearch(
    build_kt_model, 
    objective='val_loss', 
    max_trials=5,
    overwrite=True, 
    directory='kt_dir', 
    project_name='reg_tuning_adam'
)

tuner.search(X_train_sc, y_train, validation_split=0.2, epochs=20, batch_size=64, verbose=0)

best_model_kt = tuner.get_best_models(num_models=1)[0]

history_kt = best_model_kt.fit(X_train_sc, y_train, epochs=50, batch_size=64, validation_split=0.2, verbose=0)

y_pred7 = best_model_kt.predict(X_test_sc).flatten()

print('ML6 - Neural Network (KerasTuner)')
print('R2:', r2_score(y_test, y_pred7))

best_model_kt.save('models/ml6_model.h5')




136/136 [==============================] - 0s 768us/step
ML6 - Neural Network (KerasTuner)
R2: 0.8164609219369461


c:\Users\DaniilMS\miniconda3\envs\lab4_1\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
